[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JulesMalin/isba2411-nlp/blob/main/Week%202/ex1_tokenizers.ipynb)

# ISBA 2411 · Natural Language Processing & AI
## Week 2 · Lecture 3 · Exercise 1: Tokenizer Face-off
**Summer 2026 · ~15 minutes · work in pairs · no API key needed**

---

### Reading
Aligned to the current course texts:

- **J&M** (3rd-ed. draft, Jan 6 2026) -- Ch. 2 *Words and Tokens*, sec. 2.4 Byte-Pair Encoding, pp. 10-14. In this draft Ch. 2 is titled *Words and Tokens* and tokenization lives in sec. 2.4.
- **HOLLM** -- Ch. 2 *Tokens and Token Embeddings*.
- **Tunstall** (Revised Edition) -- Ch. 2, *Subword Tokenization*; Ch. 4 for the multilingual / SentencePiece view.

### What you will do
- Load the GPT-2 and BERT tokenizers from Hugging Face
- Tokenize the same four strings with each tokenizer
- Compare token counts and the actual pieces
- Find the one string where the two tokenizers disagree most

### What to bring back to the room
- Which input was most expensive in tokens?
- Did the emoji survive as one piece or shatter into many?
- What does that imply for non-English users?

### Step 0. Install and import
Run this once. It takes about 30 seconds in Colab.

In [1]:
!pip install -q transformers

from transformers import AutoTokenizer
import pandas as pd

### Step 1. Load two tokenizers
GPT-2 uses byte-level BPE. BERT uses WordPiece. Same idea, different rules, and we will see the difference.

In [2]:
gpt2_tok = AutoTokenizer.from_pretrained("gpt2")
bert_tok = AutoTokenizer.from_pretrained("bert-base-uncased")

print("GPT-2 vocab size:", gpt2_tok.vocab_size)
print("BERT  vocab size:", bert_tok.vocab_size)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

GPT-2 vocab size: 50257
BERT  vocab size: 30522


### Step 2. The four test strings
One common word, one rare word, one emoji, and one Spanish phrase. Predict in your pair which one will cost the most tokens before you run it.

In [3]:
samples = {
    "common word":  "language",
    "rare word":    "antidisestablishmentarianism",
    "emoji":        "I love NLP 🧠🔥",
    "spanish":      "El procesamiento del lenguaje natural es fascinante",
}

### Step 3. Tokenize with both and compare
`tokenize` shows the human-readable pieces. The `##` marks in BERT mean *this piece continues the previous word*. The `Ġ` (shown as a leading space) in GPT-2 marks *this piece starts a new word*.

In [4]:
def compare(label, text):
    g = gpt2_tok.tokenize(text)
    b = bert_tok.tokenize(text)
    return {
        "input": label,
        "gpt2_tokens": len(g),
        "bert_tokens": len(b),
        "gpt2_pieces": " | ".join(g),
        "bert_pieces": " | ".join(b),
    }

rows = [compare(k, v) for k, v in samples.items()]
df = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", None)
df[["input", "gpt2_tokens", "bert_tokens"]]

,input,gpt2_tokens,bert_tokens
0,common word,1,1
1,rare word,5,8
2,emoji,10,5
3,spanish,16,16


See the actual pieces side by side:

In [5]:
for r in rows:
    print(f"\n=== {r['input']} ===")
    print("GPT-2:", r["gpt2_pieces"])
    print("BERT :", r["bert_pieces"])


=== common word ===
GPT-2: language
BERT : language

=== rare word ===
GPT-2: ant | idis | establishment | arian | ism
BERT : anti | ##dis | ##est | ##ab | ##lish | ##ment | ##arian | ##ism

=== emoji ===
GPT-2: I | Ġlove | ĠN | LP | ĠðŁ | § | ł | ðŁ | Ķ | ¥
BERT : i | love | nl | ##p | [UNK]

=== spanish ===
GPT-2: El | Ġpro | ces | am | ient | o | Ġdel | Ġl | engu | aj | e | Ġnatural | Ġes | Ġfasc | in | ante
BERT : el | pro | ##ces | ##ami | ##ent | ##o | del | len | ##gua | ##je | natural | es | fa | ##sc | ##ina | ##nte


### Step 4. Your turn
Add at least two strings of your own to the dictionary below. Try to find the input where GPT-2 and BERT disagree most on token count. Ideas: a URL, a hashtag, a long number, a word in a third language, a code snippet.

In [7]:
my_samples = {
    "your string 1": "The event happens on 999999999999 at https://example.com/status/123",
    "your string 2": "#DeepLearning_2026_🔥 {x += 1;}",
}

my_rows = [compare(k, v) for k, v in my_samples.items()]
my_df = pd.DataFrame(my_rows)
my_df["abs_gap"] = (my_df["gpt2_tokens"] - my_df["bert_tokens"]).abs()
my_df.sort_values("abs_gap", ascending=False)[
    ["input", "gpt2_tokens", "bert_tokens", "abs_gap"]]

,input,gpt2_tokens,bert_tokens,abs_gap
0,your string 1,18,26,8
1,your string 2,15,17,2


### Debrief
Discuss with your pair, then be ready to report out:

1. Which input was most expensive in tokens, and why?
2. Did the emoji survive as one token or break into several bytes?
3. The Spanish phrase: how many more tokens than its English length would suggest? What does that cost a Spanish-speaking user on a per-token billed product?

---
*Notebook lives in github.com/JulesMalin/isba2411-nlp  (Week 2/). If anything breaks, flag it and we will debug together.*

In [10]:
for r in my_rows:
    print(f"\n=== {r['input']} ===")
    print("GPT-2:", r["gpt2_pieces"])
    print("BERT :", r["bert_pieces"])


=== your string 1 ===
GPT-2: The | Ġevent | Ġhappens | Ġon | Ġ9 | 9999 | 9999 | 999 | Ġat | Ġhttps | :// | example | . | com | / | status | / | 123
BERT : the | event | happens | on | 999 | ##9 | ##9 | ##9 | ##9 | ##9 | ##9 | ##9 | ##9 | ##9 | at | https | : | / | / | example | . | com | / | status | / | 123

=== your string 2 ===
GPT-2: # | Deep | Learning | _ | 20 | 26 | _ | ðŁ | Ķ | ¥ | Ġ{ | x | Ġ+= | Ġ1 | ;}
BERT : # | deep | ##lea | ##rn | ##ing | _ | 202 | ##6 | _ | [UNK] | { | x | + | = | 1 | ; | }
